# 09 — Ensembles, Protocol, Interpretation and the Decision Layer

Everything that spans both problems: whether the third model family earns its
place, what the cross-validation protocol is actually telling us, what the models
rely on, where they fail, and what the output is worth in reais.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config, evaluate as E, modeling as M
from data_load import load_all, load_analysis
from features import *
from viz import use_report_style, save_fig

use_report_style()
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
RESULTS = ROOT / "report" / "results"
def results(name):
    """Read one canonical result table written by src/train_models.py."""
    p = RESULTS / name
    assert p.exists(), f"{p} missing - run `python src/train_models.py` first"
    return pd.read_csv(p)
print("seed", config.SEED, "| window", config.ANALYSIS_START.date(), "->", config.ANALYSIS_END.date())

seed 42 | window 2017-01-01 -> 2018-08-31


## 9.1 Family C: does the ensemble earn its place?

Voting and Stacking are built from the winners of the linear and tree families.
The brief encourages ensembles *"when they add demonstrable value"*, which is a
condition, not an instruction — so the result is reported whichever way it lands.

In [2]:
reg, low = results("reg_p1a.csv"), results("clf_p2.csv")
for name, tbl, metric, better in [("P1A lead time", reg, "mae", "lower"),
                                  ("P2 low review", low, "pr_auc", "higher")]:
    print(f"\n{name} ({metric}, {better} is better)")
    for fam in ("Linear", "Tree-based", "Ensemble"):
        part = tbl[tbl["family"] == fam]
        if part.empty:
            continue
        best = part[metric].min() if better == "lower" else part[metric].max()
        row = part.loc[part[metric].idxmin() if better == "lower" else part[metric].idxmax()]
        print(f"  {fam:11s} best {best:.3f}   ({row['model']})")


P1A lead time (mae, lower is better)
  Linear      best 4.181   (LinearRegression)
  Tree-based  best 4.141   (HistGradientBoostingRegressor)
  Ensemble    best 3.898   (VotingRegressor (A+B))

P2 low review (pr_auc, higher is better)
  Linear      best 0.291   (LogisticRegression (tuned L2))
  Tree-based  best 0.415   (HistGradientBoostingClassifier)
  Ensemble    best 0.414   (StackingClassifier (A+B))


In [3]:
# The verdict, computed rather than asserted.
import json
meta = json.loads((config.MODELS_DIR / "model_metadata.json").read_text("utf-8"))
for name, tbl, key, metric, better in [
        ("P1A lead time", reg, "p1a", "mae", "lower"),
        ("P2 low review", low, "p2", "pr_auc", "higher")]:
    chosen = meta[key]["label"]
    fam = tbl.loc[tbl["model"] == chosen, "family"].iloc[0]
    best_non_ens = tbl[(tbl["family"] != "Ensemble") & (tbl["family"] != "Baseline")]
    b = (best_non_ens[metric].min() if better == "lower" else best_non_ens[metric].max())
    c = tbl.loc[tbl["model"] == chosen, metric].iloc[0]
    verdict = ("an ensemble was selected" if fam == "Ensemble"
               else "no ensemble was selected")
    print(f"{name}: {verdict} ({chosen})")
    print(f"   selected {metric} {c:.3f}  vs best single model {b:.3f}  "
          f"({'better' if (c < b) == (better == 'lower') else 'no better'})")

P1A lead time: an ensemble was selected (VotingRegressor (A+B))
   selected mae 3.898  vs best single model 4.141  (better)
P2 low review: an ensemble was selected (StackingClassifier (A+B))
   selected pr_auc 0.414  vs best single model 0.415  (no better)


**How to read that.** The ensemble is built from the winners of families A
and B and is judged on exactly the same six-window rule as everything else, so
whether it appears in the final models is a measurement, not a preference. Two
things are worth stating regardless of which way it lands.

*Why a blend can help here.* The two families fail differently under drift. Lead
times in the test window are systematically shorter than anything common in
training, and a tree cannot predict outside the range of the targets it was
fitted on — every leaf is an average of training values. A linear model has no
such bound and extrapolates the level. Averaging is then not a generic ensembling
win but one member covering the other's specific failure, which is why the
weights are 1:2 in the tree's favour rather than tuned.

*Why stacking is handicapped.* Its meta-features come from `cross_val_predict`,
which requires the inner folds to *partition* the training set. An expanding
window partitions nothing, so the inner CV has to be contiguous `KFold` and the
meta-learner is fitted on predictions partly made by models trained on later
orders. Stacking is therefore mildly *flattered* by its own protocol — worth
knowing whichever way the comparison lands.

## 9.2 The cross-validation protocol, and a trap inside PR-AUC

Three numbers for the same model: expanding-window CV, shuffled customer-disjoint
CV, and the forward test window. The raw scores disagree badly. Normalised by each
period's no-skill floor, they agree almost exactly.

In [4]:
cvp = results("cv_protocol.csv")
display(cvp[["protocol", "mean", "std", "floor", "lift"]].round(3))

t = cvp[cvp["protocol"].str.contains("TimeSeries")].iloc[0]
g = cvp[cvp["protocol"].str.contains("Group")].iloc[0]
f = cvp[cvp["protocol"].str.contains("Forward")].iloc[0]
print(f"\nraw PR-AUC : CV {t['mean']:.3f} / {g['mean']:.3f} vs a forward {f['mean']:.3f}")
print(f"floors     : {t['floor']:.3f} / {g['floor']:.3f} vs {f['floor']:.3f}")
print(f"lift       : {t['lift']:.2f}x / {g['lift']:.2f}x vs {f['lift']:.2f}x")
print(f"\nfold spread: temporal +/-{t['std']:.3f}, shuffled +/-{g['std']:.3f} "
      f"({t['std']/g['std']:.0f}x narrower)")

import figures_phase2 as F
print(Path(F.fig_cv_protocol()).name)

,protocol,mean,std,floor,lift
0,TimeSeriesSplit (expanding window),0.562,0.090,0.158,3.551
1,StratifiedGroupKFold (shuffled),0.580,0.006,0.158,3.664
2,Forward test window (the truth),0.414,NaN,0.110,3.751



raw PR-AUC : CV 0.562 / 0.580 vs a forward 0.414
floors     : 0.158 / 0.158 vs 0.110
lift       : 3.55x / 3.66x vs 3.75x

fold spread: temporal +/-0.090, shuffled +/-0.006 (16x narrower)


fig16_cv_protocol.pdf


Two things fall out of this table.

**Raw PR-AUC is not comparable across periods with different base rates.** Average
precision has a no-skill floor equal to the positive rate, and the low-review rate
falls from the training period to the test period. Most of the apparent collapse
from CV to forward test is that mechanical shift, not a modelling failure. The
*lift* over the floor is the quantity that transfers.

**The shuffled protocol reports a far tighter interval than it is entitled to.**
Its fold-to-fold spread is a small fraction of the temporal one, because every
fold is drawn from the same mixed period. It looks more precise while being no
closer to the forward truth — the most dangerous combination a validation
protocol can offer.

## 9.3 What the models actually rely on

Permutation importance on the held-out window, not impurity importance from the
training set: impurity-based scores inflate high-cardinality features (Strobl et
al., 2007), and with 74 product categories in play that is not a hypothetical.

In [5]:
for tag, name in [("P1A lead time", "perm_p1a.csv"), ("P2 low review", "perm_p2.csv")]:
    perm = results(name)
    print(f"\n{tag}")
    for _, r in perm.head(8).iterrows():
        bar = "#" * int(60 * abs(r["importance"]) / abs(perm["importance"]).max())
        print(f"  {r['feature']:30s} {r['importance']:+8.4f}  {bar}")
print(Path(F.fig_importance()).name)


P1A lead time
  lead_payment_type               +0.0336  ####################
  max_volume_cm3                  +0.0210  ############
  purchase_dow                    +0.0197  ###########
  n_sellers                       +0.0189  ###########
  log_weight                      +0.0173  ##########
  seller_region                   +0.0138  ########
  n_items                         +0.0136  ########
  estimated_days                  +0.0091  #####

P2 low review
  delay_vs_estimate_days          +0.2061  ############################################################
  delivery_days                   +0.0742  #####################
  n_items                         +0.0448  #############
  n_sellers                       +0.0108  ###
  hist_route_lead_days            +0.0067  #
  hist_seller_low_review_rate     +0.0062  #
  estimated_days                  +0.0044  #
  is_late                         +0.0027  


fig17_importance.pdf


In [6]:
perm2 = results("perm_p2.csv")
top2 = perm2.head(2)["importance"].sum()
tot = perm2.loc[perm2["importance"] > 0, "importance"].sum()
print(f"the top two features carry {100*top2/tot:.0f}% of the review model's total "
      f"positive importance")
print("\nWhich is the honest description of this model: it is mostly asking how "
      "late the parcel was, and how big the order was. That is also why a "
      "depth-limited tree came so close to the boosted one.")

the top two features carry 77% of the review model's total positive importance

Which is the honest description of this model: it is mostly asking how late the parcel was, and how big the order was. That is also why a depth-limited tree came so close to the boosted one.


## 9.4 Where it fails

An aggregate score can hide a model that works for São Paulo and nowhere else.

In [7]:
for name, key, col in [("segments_p2_region.csv", "customer_region", "roc_auc"),
                       ("segments_p1a_region.csv", "customer_region", "mae")]:
    seg = results(name)
    if seg.empty:
        continue
    print(f"\n{name}")
    display(seg.round(3))
    print(f"  spread on {col}: {seg[col].min():.3f} to {seg[col].max():.3f}")
print(Path(F.fig_segments()).name)


segments_p2_region.csv


,customer_region,n,positive_rate,roc_auc,pr_auc,pr_auc_lift,balanced_accuracy,precision,recall,f1
0,Southeast,9114,0.113,0.747,0.411,3.641,0.683,0.576,0.405,0.475
1,South,1684,0.090,0.744,0.356,3.945,0.654,0.505,0.342,0.408
2,Northeast,1132,0.118,0.756,0.454,3.836,0.685,0.628,0.403,0.491
3,Central-West,720,0.115,0.761,0.494,4.283,0.694,0.614,0.422,0.500
4,North,205,0.093,0.778,0.464,5.012,0.644,0.545,0.316,0.400


  spread on roc_auc: 0.744 to 0.778

segments_p1a_region.csv


,customer_region,n,mae,bias,actual_mean
0,Southeast,8951,3.422,1.151,7.193
1,South,1657,4.349,2.720,9.234
2,Northeast,1109,6.272,4.221,13.420
3,Central-West,710,4.291,2.212,10.713
4,North,203,6.839,4.774,15.641


  spread on mae: 3.422 to 6.839


fig18_segments.pdf

## 9.5 The decision layer

A probability is not a decision. Turning one into the other needs a cost, and Olist
publishes none — so the costs below are **stated assumptions**, and the conclusion
is read as a shape with a sensitivity sweep rather than as a number
(Elkan, 2001).

In [8]:
cost = results("cost_p2.csv")
print("assumptions (R$ per order):", config.COSTS)
opt = cost.loc[cost["net_benefit"].idxmax()]
print(f"\noptimal threshold under these costs : {opt['threshold']:.2f}")
print(f"  flags {opt['flagged']:,} of {len(results('predictions_p2_test.csv')):,} orders")
print(f"  expected net benefit: R$ {opt['net_benefit_per_1k']:.0f} per 1,000 delivered orders")

print("\nsensitivity - how the optimum moves with the cost ratio:")
import numpy as np
from evaluate import cost_curve
pred = results("predictions_p2_test.csv")
y, p = pred["actual_low_review"], pred["p_low_review"]
rows = []
for ratio in (2, 4, 8, 16):
    c = cost_curve(y, p, 5.0, 5.0 * ratio)
    b = c.loc[c["net_benefit"].idxmax()]
    rows.append({"cost ratio (miss : action)": f"{ratio}:1",
                 "optimal threshold": b["threshold"],
                 "orders flagged": int(b["flagged"]),
                 "net benefit / 1k": round(b["net_benefit_per_1k"], 1)})
display(pd.DataFrame(rows))
print("\nThe threshold moves with the ratio, as it must - which is the point of "
      "showing it rather than quoting one number.")

assumptions (R$ per order): {'outreach_cost': 5.0, 'low_review_cost': 40.0, 'expedite_cost': 12.0, 'late_delivery_cost': 40.0}

optimal threshold under these costs : 0.11
  flags 1,927.0 of 12,855 orders
  expected net benefit: R$ 1466 per 1,000 delivered orders

sensitivity - how the optimum moves with the cost ratio:


,cost ratio (miss : action),optimal threshold,orders flagged,net benefit / 1k
0,2:1,0.40,709,89.1
1,4:1,0.22,940,501.0
2,8:1,0.11,1927,1466.0
3,16:1,0.08,5383,4285.1



The threshold moves with the ratio, as it must - which is the point of showing it rather than quoting one number.


## 9.6 The pre-registered bands

The expected ranges in `config.EXPECTED_BANDS` were written down before any Phase 2
model was tuned, from the Phase 1 probes. They serve as success criteria and as a
leak alarm: a result far *better* than its band is not a triumph, it is a bug —
adding one forbidden column took the Phase 1 late-delivery model from ROC-AUC 0.589
to 1.000.

In [9]:
bands = results("bands.csv")
display(bands)
suspicious = bands[bands["verdict"].str.startswith("SUSPICIOUSLY")]
assert suspicious.empty, f"investigate before reporting: {list(suspicious['metric'])}"
print(f"{(bands['verdict'] == 'in band').sum()} of {len(bands)} metrics landed inside "
      "the range registered in advance; none landed suspiciously above it.")

,metric,value,band,verdict
0,p1a_mae,3.8980,"[3.3, 4.3]",in band
1,p1b_balanced_accuracy,0.7045,"[0.58, 0.72]",in band
2,p1b_roc_auc,0.7820,"[0.72, 0.85]",in band
3,p2_balanced_accuracy,0.6806,"[0.63, 0.74]",in band
4,p2_roc_auc,0.7490,"[0.7, 0.79]",in band
5,p2_pr_auc,0.4137,"[0.34, 0.48]",in band


6 of 6 metrics landed inside the range registered in advance; none landed suspiciously above it.


## 9.7 Deployment artifacts

In [10]:
import json
meta = json.loads((config.MODELS_DIR / "model_metadata.json").read_text("utf-8"))
total = 0
for f in sorted(config.MODELS_DIR.iterdir()):
    total += f.stat().st_size
    print(f"  {f.name:30s} {f.stat().st_size/1e6:6.3f} MB")
print(f"  {'TOTAL':30s} {total/1e6:6.3f} MB")
assert total < 25e6, "artifacts too large for the deployment container"

print(f"\nserved at https://it5006-group-project-10.streamlit.app (Predict tab)")
print("round-trip check: reload each model and re-score the test window")
b = M.load_model(config.MODELS_DIR / "p2_lowreview.joblib")
pred = results("predictions_p2_test.csv")
df = build_features(load_analysis(load_all()), with_history=True)
te = df[df["order_id"].isin(pred["order_id"])].set_index("order_id").loc[pred["order_id"]]
p = b["estimator"].predict_proba(te[b["features"]])[:, 1]
print(f"  max |reloaded - saved| = {np.abs(p - pred['p_low_review'].to_numpy()).max():.2e}")
assert np.allclose(p, pred["p_low_review"].to_numpy(), atol=1e-9), \
    "the saved model does not reproduce the reported predictions"

  history_lookup.csv              0.117 MB
  model_metadata.json             0.003 MB
  p1a_leadtime.joblib             0.381 MB
  p1b_margin_to_risk.joblib       0.001 MB
  p2_lowreview.joblib             0.321 MB
  TOTAL                           0.824 MB

served at https://it5006-group-project-10.streamlit.app (Predict tab)
round-trip check: reload each model and re-score the test window


  max |reloaded - saved| = 9.71e-17
